## 3. Feature Engineering

### Objective

Create one student-level feature table using only information
available early in the course.

The engineered features will combine:
- Student demographic and academic information
- Registration information
- Early assessment performance
- Early VLE engagement

The target will identify students who eventually failed or withdrew.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

student_info = pd.read_csv(PROCESSED_DIR / "student_info_clean.csv")
student_registration = pd.read_csv(PROCESSED_DIR / "student_registration_clean.csv")
student_assessment = pd.read_csv(PROCESSED_DIR / "student_assessment_clean.csv")
assessments = pd.read_csv(PROCESSED_DIR / "assessments_clean.csv")
student_vle = pd.read_csv(PROCESSED_DIR / "student_vle_clean.csv")

## 3.1 Early Prediction Window

We define the first 28 days of the course as the early-warning window.

Only information available on or before day 28 will be used to construct
early engagement and assessment features.

This prevents later information from leaking into the early-risk model.

In [2]:
CUTOFF_DAY = 28

print("Early prediction cutoff:", CUTOFF_DAY, "days")

Early prediction cutoff: 28 days


## Insight

A fixed early cutoff creates a realistic prediction setting: the model
must identify risk using information that would actually be available
during the first few weeks of a course.

In [3]:
student_base = student_info.merge(
    student_registration,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

print("Rows:", len(student_base))
print("Unique student-course records:",
      student_base[
          ["code_module", "code_presentation", "id_student"]
      ].drop_duplicates().shape[0])

Rows: 32593
Unique student-course records: 32593


In [4]:
student_base.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,date_unregistration_missing
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,1
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,-53.0,NaN,1
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,-92.0,12.0,0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,-52.0,NaN,1
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,-176.0,NaN,1


## Insight

The base table keeps one record per student-course-presentation combination.
This granularity matches the structure of the OULAD outcome and prevents
incorrect many-to-many joins during feature construction.

In [5]:
student_base["is_at_risk"] = (
    student_base["final_result"].isin(["Fail", "Withdrawn"]).astype("Int64")
)

student_base["is_at_risk"].value_counts()

is_at_risk
1    17208
0    15385
Name: count, dtype: Int64

In [6]:
student_base["is_at_risk"].value_counts(normalize=True).round(3)

is_at_risk
1    0.528
0    0.472
Name: proportion, dtype: Float64

## Insight

The target `is_at_risk` identifies students whose final outcome was
Fail or Withdrawn.

`final_result` is used only to construct the target and will not be
included among the predictive features, preventing direct target leakage.

In [7]:
student_base["registered_before_start"] = (
    student_base["date_registration"] < 0
).astype("Int64")

## Insight

Registration timing is retained as an early structural feature.
Withdrawal date is excluded because it directly reflects a student's
final enrollment status and would leak future information into the model.

In [8]:
early_assessments = assessments[
    assessments["date"].notna()
    & (assessments["date"] <= CUTOFF_DAY)
].copy()

print("Early assessments:", len(early_assessments))

Early assessments: 17


In [9]:
early_student_assessment = student_assessment.merge(
    early_assessments[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_type",
            "date",
            "weight"
        ]
    ],
    on="id_assessment",
    how="inner"
)

In [10]:
early_student_assessment = early_student_assessment[
    early_student_assessment["date_submitted"].notna()
    & (early_student_assessment["date_submitted"] <= CUTOFF_DAY)
].copy()

In [11]:
assessment_features = (
    early_student_assessment
    .groupby(
        ["code_module", "code_presentation", "id_student"],
        as_index=False
    )
    .agg(
        early_assessments_taken=("id_assessment", "nunique"),
        early_mean_score=("score", "mean"),
        early_median_score=("score", "median"),
        early_max_score=("score", "max"),
        early_score_std=("score", "std"),
        early_missing_scores=("is_score_missing", "sum"),
        early_assessment_weight=("weight", "sum")
    )
)

In [12]:
student_base = student_base.merge(
    assessment_features,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

## Insight

Early assessment features summarize the student's academic performance
using only assessments available within the early-warning window.

Students without an early assessment record are not automatically treated
as having failed; missing assessment activity is preserved for later
interpretation.

In [13]:
early_vle = student_vle[
    student_vle["date"] <= CUTOFF_DAY
].copy()

print("Early VLE records:", len(early_vle))

Early VLE records: 2672327


In [14]:
vle_features = (
    early_vle
    .groupby(
        ["code_module", "code_presentation", "id_student"],
        as_index=False
    )
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        active_sites=("id_site", "nunique")
    )
)

In [15]:
vle_features["avg_clicks_per_active_day"] = (
    vle_features["total_clicks"]
    / vle_features["active_days"].replace(0, np.nan)
)

In [16]:
student_base = student_base.merge(
    vle_features,
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

In [17]:
student_base["no_early_vle_activity"] = (
    student_base["total_clicks"].isna()
).astype("Int64")

## Insight

Early VLE engagement is represented through total clicks, active days,
and active learning sites.

Students with no recorded early VLE activity are preserved rather than
removed, because lack of activity may itself be an important early-warning
signal.

In [18]:
target = "is_at_risk"

excluded_columns = [
    "final_result",
    "date_unregistration",
    "date_unregistration_missing"
]

model_features = student_base.drop(
    columns=excluded_columns,
    errors="ignore"
).copy()

In [19]:
print("Rows:", len(model_features))
print("Columns:", len(model_features.columns))

model_features.head()

Rows: 32593
Columns: 26


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,is_at_risk,registered_before_start,early_assessments_taken,early_mean_score,early_median_score,early_max_score,early_score_std,early_missing_scores,early_assessment_weight,total_clicks,active_days,active_sites,avg_clicks_per_active_day,no_early_vle_activity
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,0,1,1.0,78.0,78.0,78.0,NaN,0.0,10.0,400.0,8.0,24.0,50.000000,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,0,1,1.0,70.0,70.0,70.0,NaN,0.0,10.0,609.0,19.0,34.0,32.052632,0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.0,12.0,22.0,21.666667,0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,0,1,1.0,72.0,72.0,72.0,NaN,0.0,10.0,469.0,22.0,31.0,21.318182,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,-176.0,0,1,1.0,69.0,69.0,69.0,NaN,0.0,10.0,559.0,24.0,34.0,23.291667,0


In [20]:
print(
    "Unique student-course records:",
    model_features[
        ["code_module", "code_presentation", "id_student"]
    ].drop_duplicates().shape[0]
)

Unique student-course records: 32593


In [21]:
model_features[target].value_counts(dropna=False)

is_at_risk
1    17208
0    15385
Name: count, dtype: Int64

## 3.2 Leakage Check

We verify that the final feature table does not contain direct outcome
information or variables derived from events occurring after the
early-prediction cutoff.

Assessment and VLE features were already restricted to information
available on or before day 28.

In [22]:
leakage_columns = [
    col for col in [
        "final_result",
        "date_unregistration",
        "date_unregistration_missing"
    ]
    if col in model_features.columns
]

print("Potential leakage columns:", leakage_columns)

Potential leakage columns: []


## Insight

The final feature table excludes final outcome information and withdrawal
timing from the predictors. This ensures that the early-risk model uses
only information that could have been available before the prediction
cutoff.

In [23]:
FEATURE_DIR = BASE_DIR / "data" / "processed"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

output_path = FEATURE_DIR / "early_risk_features.csv"

model_features.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: d:\Early Risk Detection for Student Success - OULAD\data\processed\early_risk_features.csv


## Feature Engineering Summary

The feature engineering process created a student-level early-risk dataset
by combining demographic, academic, registration, assessment, and VLE
information.

Key design decisions:

- One row per student-course-presentation
- Early prediction cutoff at day 28
- Assessment features restricted to early assessments
- VLE engagement restricted to early activity
- Missing activity preserved as meaningful information
- Final outcome excluded from predictors
- Withdrawal date excluded to prevent target leakage

The resulting dataset is ready for exploratory analysis and model
development.